# HybridBayesNet

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/hybrid/doc/HybridBayesNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --quiet gtsam

A `HybridBayesNet` represents a directed graphical model (Bayes Net) specifically designed for hybrid systems. It is a collection of `gtsam.HybridConditional` objects, ordered according to an elimination sequence.

It extends `gtsam.BayesNet<HybridConditional>` and allows representing the joint probability distribution $P(X, M)$ over continuous variables $X$ and discrete variables $M$ as a product of conditional probabilities:
$$
P(X, M) = \prod_i P(\text{Frontal}_i | \text{Parents}_i)
$$
where each conditional $P(\text{Frontal}_i | \text{Parents}_i)$ is stored as a `HybridConditional`. This structure allows for representing complex dependencies, such as continuous variables conditioned on discrete modes ($P(X|M)$) alongside purely discrete ($P(M)$) or purely continuous ($P(X)$) relationships.

`HybridBayesNet` objects are typically obtained by eliminating a `HybridGaussianFactorGraph`.

In [3]:
import gtsam
print([k for k in gtsam.__dict__.keys() if "Discrete" in k])
from gtsam import DiscreteKeys
help(DiscreteKeys.push_back)

['DiscreteValues', 'DiscreteKeys', 'DiscreteFactor', 'DiscreteConditional', 'DiscreteDistribution', 'DiscreteBayesNet', 'DiscreteBayesTreeClique', 'DiscreteBayesTree', 'DiscreteLookupTable', 'DiscreteLookupDAG', 'DiscreteFactorGraph', 'DiscreteEliminationTree', 'DiscreteCluster', 'DiscreteJunctionTree', 'DiscreteSearchSolution', 'DiscreteSearch', 'PrintDiscreteValues', 'EliminateDiscrete']
Help on instancemethod in module gtsam.gtsam:

push_back(...)
    push_back(self: gtsam.gtsam.DiscreteKeys, point_pair: tuple[int, int]) -> None



In [9]:
import gtsam
import numpy as np

from gtsam import (
    HybridConditional, GaussianConditional, DiscreteConditional, HybridGaussianConditional,
    HybridGaussianFactorGraph, HybridGaussianFactor, JacobianFactor, DecisionTreeFactor,
    DiscreteValues, VectorValues, HybridValues, Ordering,
    HybridBayesNet, GaussianBayesNet, DiscreteBayesNet
)
from gtsam.symbol_shorthand import X, D

## Creating a HybridBayesNet

While they can be constructed manually by adding `HybridConditional`s, they are more commonly obtained via elimination of a `HybridGaussianFactorGraph`.

In [10]:
# --- Method 1: Manual Construction ---
# hbn_manual = gtsam.HybridBayesNet()

# P(D0)
dk0 = (D(0), 2)
cond_d0 = DiscreteConditional(dk0, [], "7/3") # P(D0=0)=0.7
# hbn_manual.push_back(HybridConditional(cond_d0))

# # P(X0 | D0)
# dk0_parent = (D(0), 2)
#  # Mode 0: P(X0 | D0=0) = N(0, 1)
# gc0 = GaussianConditional(X(0), np.zeros(1), np.eye(1), gtsam.noiseModel.Unit.Create(1))
#  # Mode 1: P(X0 | D0=1) = N(5, 4)
# gc1 = GaussianConditional(X(0), np.array([2.5]), np.eye(1)*0.5, gtsam.noiseModel.Isotropic.Sigma(1,2.0))
# cond_x0_d0 = HybridGaussianConditional(dk0_parent, [gc0, gc1])
# hbn_manual.push_back(HybridConditional(cond_x0_d0))

# # P(X1 | X0)
# cond_x1_x0 = GaussianConditional(X(1), np.array([0.0]), np.eye(1), # d, R=I
#                              X(0), np.eye(1),                  # Parent X0, S=I
#                              gtsam.noiseModel.Isotropic.Sigma(1, 1.0)) # N(X1; X0, I)
# hbn_manual.push_back(HybridConditional(cond_x1_x0))

# print("Manually Constructed HybridBayesNet:")
# hbn_manual.print()

# --- Method 2: From Elimination ---
hgfg = HybridGaussianFactorGraph()
# P(D0) = 70/30
hgfg.add(DecisionTreeFactor([dk0], "0.7 0.3"))
# P(X0|D0) = mixture N(0,1); N(5,4)
# Factor version: 0.5*|X0-0|^2/1 + C0 ; 0.5*|X0-5|^2/4 + C1
factor_gf0 = JacobianFactor(X(0), np.eye(1), np.zeros(1), gtsam.noiseModel.Isotropic.Sigma(1, 1.0))
factor_gf1 = JacobianFactor(X(0), np.eye(1), np.array([5.0]), gtsam.noiseModel.Isotropic.Sigma(1, 2.0))
# Store -log(prior) for D0 in the hybrid factor (optional, could keep separate)
logP_D0_0 = -np.log(0.7)
logP_D0_1 = -np.log(0.3)
hgfg.add(HybridGaussianFactor(dk0, [(factor_gf0, logP_D0_0), (factor_gf1, logP_D0_1)]))
# P(X1|X0) = N(X0, 1)
hgfg.add(JacobianFactor(X(0), -np.eye(1), X(1), np.eye(1), np.zeros(1), gtsam.noiseModel.Isotropic.Sigma(1, 1.0)))

print("\nOriginal HybridGaussianFactorGraph for Elimination:")
hgfg.print()

ordering = Ordering([X(0), X(1), D(0)]) # Eliminate continuous first
# Note: Using HybridOrdering(hgfg) is generally recommended
# ordering = gtsam.HybridOrdering(hgfg)


hbn_elim, _ = hgfg.eliminatePartialSequential(ordering)
print("\nHybridBayesNet from Elimination:")
hbn_elim.print()
# Note: Elimination details might differ slightly from manual construction
# depending on how factors combine. The represented joint P(X,M) should be proportional.

AttributeError: 'gtsam.gtsam.HybridGaussianFactorGraph' object has no attribute 'add'

## Operations on HybridBayesNet

`HybridBayesNet` allows evaluating the joint probability, sampling, optimizing (finding the MAP state), and extracting marginal or conditional distributions.

In [11]:
# Use the Bayes Net from elimination for consistency
hbn = hbn_elim

# --- Evaluation ---
values = gtsam.HybridValues()
values.insert(D(0), 0)
values.insert(X(0), np.array([0.1]))
values.insert(X(1), np.array([0.2]))

log_prob = hbn.logProbability(values)
prob = hbn.evaluate(values) # Same as exp(log_prob)
print(f"\nLogProbability P(X0=0.1, X1=0.2, D0=0): {log_prob}")
print(f"Probability P(X0=0.1, X1=0.2, D0=0): {prob}")

# --- Sampling ---
# Ancestral sampling requires a random number generator
rng = np.random.default_rng(42)
# Convert numpy RNG to std::mt19937_64 (GTSAM may need specific type)
# This part might require a C++ helper or a different RNG setup in practice.
# Using default GTSAM RNG for now.
# sample = hbn.sample(values) # Sample given certain values
try:
    full_sample = hbn.sample()
    print("\nSampled HybridValues:")
    full_sample.print()
except Exception as e:
    print(f"\nSampling error (may require std::mt19937_64): {e}")


# --- Optimization (Finding MAP state) ---
# Computes MPE for discrete, then optimizes continuous given MPE
map_solution = hbn.optimize()
print("\nMAP Solution (Optimize):")
map_solution.print()

# --- MPE (Most Probable Explanation for Discrete Variables) ---
mpe_assignment = hbn.mpe()
print("\nMPE Discrete Assignment:")
mpe_assignment.print() # Should match discrete part of map_solution

# --- Optimize Continuous given specific Discrete Assignment ---
cont_solution_d0_eq_1 = hbn.optimize(gtsam.DiscreteValues([(D(0), 1)]))
print("\nOptimized Continuous Solution for D0=1:")
cont_solution_d0_eq_1.print()

# --- Extract Marginal/Conditional Distributions ---
# Get P(M) = P(D0)
discrete_marginal_bn = hbn.discreteMarginal()
print("\nDiscrete Marginal P(M):")
discrete_marginal_bn.print()

# Get P(X | M=m) = P(X0, X1 | D0=0)
gaussian_conditional_bn = hbn.choose(gtsam.DiscreteValues([(D(0), 0)]))
print("\nGaussian Conditional P(X | D0=0):")
gaussian_conditional_bn.print()

NameError: name 'hbn_elim' is not defined

## Advanced Operations (`errorTree`, `discretePosterior`, `prune`)

In [ ]:
# --- Error Tree (Log P'(M|x) = log P(x|M) + log P(M)) ---
# Evaluate unnormalized log posterior of discrete modes given continuous values
cont_values_for_error = gtsam.VectorValues()
cont_values_for_error.insert(X(0), np.array([0.5]))
cont_values_for_error.insert(X(1), np.array([1.0]))

error_tree = hbn.errorTree(cont_values_for_error)
print("\nError Tree (Unnormalized Log Posterior Log P'(M|x)) for x0=0.5, x1=1.0:")
error_tree.print()

# --- Discrete Posterior P(M|x) ---
# Normalized version of exp(-errorTree)
posterior_tree = hbn.discretePosterior(cont_values_for_error)
print("\nDiscrete Posterior Tree P(M|x) for x0=0.5, x1=1.0:")
posterior_tree.print()

# --- Pruning ---
# Reduces complexity by removing low-probability discrete branches
max_leaves = 1 # Force pruning to the most likely mode
fixed_values = gtsam.DiscreteValues() # To store values fixed during pruning
pruned_hbn = hbn.prune(max_leaves, fixedValues=fixed_values)

print(f"\nPruned HybridBayesNet (max_leaves={max_leaves}):")
pruned_hbn.print()
print("\nFixed values during pruning:")
fixed_values.print()